# M5 — SHAP Explainability Analysis
## Anemia CDS Project — Module 5

| Info | Value |
|---|---|
| **Project** | Anemia Clinical Decision Support System |
| **Module** | M5 — SHAP Explainability |
| **Input** | Frozen Models (4×AutoGluon), Calibrated Probabilities |
| **Output** | SHAP values, figures, Excel tables, `src/m5_shap.py` |
| **Depends on** | M1 (Calibration), M4 (E2E Pipeline) |

---

### Table of Contents (TOC)

1. **Setup & Import**
2. **Directory Structure & Constants**
3. **Model & Data Loading**
4. **SHAP Computation**
   - 4.1 Function Definitions
   - 4.2 OOF SHAP (4 Models)
   - 4.3 Test SHAP (Validation)
5. **SHAP Analyses & Outputs**

# 1. Setup & Import

In [ ]:
# --- Environment Setup ---
# This notebook was developed in Google Colab with Google Drive.
# To reproduce: mount your data directory and adjust paths in src/cds_config.py.
# See README.md for the expected directory structure.
#
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# CELL 1 — pip installs
!pip install shap --quiet
!pip install autogluon.tabular --quiet

In [ ]:
# CELL 2 — Import & Notebook Setup
import importlib, sys, os
import warnings
warnings.filterwarnings('ignore')

src_dir = '/content/drive/MyDrive/0000_Kaan_CDS/src'
os.chdir(src_dir)
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

for mod_name in ['plot_style', 'cds_config', 'm1_calibration', 'm2_thresholds', 'm3_errors', 'm4_e2e']:
    spec = importlib.util.spec_from_file_location(
        mod_name, os.path.join(src_dir, f'{mod_name}.py'))
    mod = importlib.util.module_from_spec(spec)
    sys.modules[mod_name] = mod
    spec.loader.exec_module(mod)

from cds_config import *
setup_notebook('Chat 5 — M5 SHAP Explainability')
from m1_calibration import load_calibrated_probs
from m4_e2e import compute_missing_ratios

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
import shap
import joblib
import pickle
from pathlib import Path
from autogluon.tabular import TabularPredictor
from collections import OrderedDict

print('✅ Import complete')
print(f'SHAP version: {shap.__version__}')

# 2. Directory Structure & Constants

In [ ]:
# CELL 3 — Directories, color palette, feature mapping, thesis references

# ── Directories ──
MOD_DIR      = PATHS.module_dir('m5_shap')
PLOT_DIR     = PATHS.module_dir('m5_shap', 'plots')
ANALYSIS_DIR = MOD_DIR / 'analysis'
SHAP_DIR     = MOD_DIR / 'shap_values'
REPORT_DIR   = MOD_DIR / 'reports'

for d in [MOD_DIR, PLOT_DIR, ANALYSIS_DIR, SHAP_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Color Palette ──
CLASS_COLORS_S2 = {
    'DEA':      PALETTE['highlight'],
    'HA':       PALETTE['base1'],
    'HGB HTZ':  PALETTE['accent1'],
    'Normal':   PALETTE['accent2'],
}

SCENARIO_COLORS = {
    'CBC_Only': PALETTE['base1'],
    'CBC_BIO':  PALETTE['accent3'],
}

cmap_tufte = LinearSegmentedColormap.from_list(
    'tufte_cm', ['#FFFFFF', PALETTE['base1']], N=256)

# ── Feature Display Mapping ──
FEATURE_DISPLAY = {
    'mcv_f_l': 'MCV', 'ret_he_pg': 'Ret-He', 'rbc_10_6_u_l': 'RBC',
    'ret_number_10_6_l': 'RET#', 'rdw_sd_fl': 'RDW-SD',
    'delta_he_pg': 'Delta-He', 'frc_perc': 'FRC', 'mchc_g_dl': 'MCHC',
    'irf_pct': 'IRF', 'nrbc_pct': 'NRBC', 'hgb_g_d_l': 'HGB',
    'micro_macro_ratio': 'MicroR/MacroR', 'yas': 'Age',
    'ferritin': 'Ferritin', 'demir': 'Iron', 'ldh': 'LD', 'uibc': 'UIBC',
    'hgb_g_d_l_div_mcv_f_l': 'HGB/MCV',
    'hgb_g_d_l_div_ret_he_pg': 'HGB/Ret-He',
    'rbc_10_6_u_l_div_mcv_f_l': 'RBC/MCV',
    'rbc_10_6_u_l_div_ret_he_pg': 'RBC/Ret-He',
    'irf_pct_div_micro_macro_ratio': 'IRF/(MicroR/MacroR)',
    'hgb_g_d_l_div_rdw_sd_fl': 'HGB/RDW-SD',
    'rbc_10_6_u_l_div_rdw_sd_fl': 'RBC/RDW-SD',
    'hgb_g_d_l_div_mchc_g_dl': 'HGB/MCHC',
    'mcv_f_l_div_rdw_sd_fl': 'MCV/RDW-SD',
    'rdw_sd_fl_div_micro_macro_ratio': 'RDW-SD/(MicroR/MacroR)',
    'ret_he_pg_div_micro_macro_ratio': 'Ret-He/(MicroR/MacroR)',
    'ret_number_10_6_l_div_ret_he_pg': 'RET#/Ret-He',
    'mchc_g_dl_div_rdw_sd_fl': 'MCHC/RDW-SD',
    'mcv_f_l_div_micro_macro_ratio': 'MCV/(MicroR/MacroR)',
    'hgb_g_d_l_div_rbc_10_6_u_l': 'HGB/RBC',
    'ret_number_10_6_l_div_mcv_f_l': 'RET#/MCV',
    'ret_number_10_6_l_div_mchc_g_dl': 'RET#/MCHC',
    'rbc_10_6_u_l_div_mchc_g_dl': 'RBC/MCHC',
    'rdw_sd_fl_div_ferritin': 'RDW-SD/Ferritin',
    'mchc_g_dl_div_ferritin': 'MCHC/Ferritin',
    'mcv_f_l_div_ferritin': 'MCV/Ferritin',
    'rbc_10_6_u_l_div_ferritin': 'RBC/Ferritin',
    'ret_he_pg_div_ferritin': 'Ret-He/Ferritin',
    'hgb_g_d_l_div_ferritin': 'HGB/Ferritin',
    'ret_he_pg_div_demir': 'Ret-He/Iron',
    'mcv_f_l_div_demir': 'MCV/Iron',
    'mchc_g_dl_div_demir': 'MCHC/Iron',
    'rdw_sd_fl_div_demir': 'RDW-SD/Iron',
    'rbc_10_6_u_l_div_demir': 'RBC/Iron',
    'hgb_g_d_l_div_demir': 'HGB/Iron',
    'ferritin_div_uibc': 'Ferritin/UIBC',
    'demir_div_uibc': 'Iron/UIBC',
    'ldh_div_uibc': 'LD/UIBC',
    'demir_div_ldh': 'Iron/LD',
}

def feat_display(name):
    return FEATURE_DISPLAY.get(name, name)

def feat_display_list(names):
    return [feat_display(n) for n in names]

# ── Feature Importance Ranking (Reference) ──
THESIS_IMPORTANCE = {
    ('1', 'CBC_Only'): ['yas', 'hgb_g_d_l_div_mcv_f_l', 'rbc_10_6_u_l',
                         'hgb_g_d_l_div_ret_he_pg', 'irf_pct_div_micro_macro_ratio'],
    ('1', 'CBC_BIO'):  ['demir', 'uibc', 'rdw_sd_fl', 'yas', 'ldh'],
    ('2', 'CBC_Only'): ['yas', 'mchc_g_dl', 'irf_pct', 'delta_he_pg', 'frc_perc'],
    ('2', 'CBC_BIO'):  ['yas', 'ldh', 'ldh_div_uibc', 'frc_perc', 'uibc'],
}

# ── Biochemistry Feature List ──
BIO_FEATURES = [
    'ferritin', 'demir', 'ldh', 'uibc',
    'rdw_sd_fl_div_ferritin', 'mchc_g_dl_div_ferritin',
    'mcv_f_l_div_ferritin', 'rbc_10_6_u_l_div_ferritin',
    'ret_he_pg_div_ferritin', 'hgb_g_d_l_div_ferritin',
    'ret_he_pg_div_demir', 'mcv_f_l_div_demir',
    'mchc_g_dl_div_demir', 'rdw_sd_fl_div_demir',
    'rbc_10_6_u_l_div_demir', 'hgb_g_d_l_div_demir',
    'ferritin_div_uibc', 'demir_div_uibc',
    'ldh_div_uibc', 'demir_div_ldh',
]

# ── Panel Constants ──
PANEL_TITLES = {
    ('1', 'CBC_Only'): 'S1 — CBC Only',
    ('1', 'CBC_BIO'):  'S1 — CBC + Biochemistry',
    ('2', 'CBC_Only'): 'S2 — CBC Only',
    ('2', 'CBC_BIO'):  'S2 — CBC + Biochemistry',
}

PANEL_POS = {
    ('1', 'CBC_Only'): (0, 0), ('1', 'CBC_BIO'): (0, 1),
    ('2', 'CBC_Only'): (1, 0), ('2', 'CBC_BIO'): (1, 1),
}

print(f'✅ Directories successfully created.')
print(f'   MOD_DIR : {MOD_DIR}')
print(f'   PLOT_DIR: {PLOT_DIR}')
print(f'   SHAP_DIR: {SHAP_DIR}')

# 3. Model & Data Loading

In [ ]:
# CELL 4 — load 4 models + OOF/test data

def load_model_and_data(stage, scenario, split='oof'):
    pattern = f'*Stage{stage}_{scenario}*'
    model_dir = list(PATHS.frozen_models.glob(pattern))[0]
    model = TabularPredictor.load(str(model_dir / 'AutoGluon_Model'))
    features = joblib.load(model_dir / 'feature_names.joblib')
    class_map = None
    cm_path = model_dir / 'class_mapping.joblib'
    if cm_path.exists():
        class_map = joblib.load(cm_path)
    df = load_calibrated_probs(PATHS, stage, scenario, split)
    df = compute_missing_ratios(df, features)
    X = df[features].copy()
    y = df['target'].values
    return model, features, class_map, df, X, y

# --- Load all models ---
MODELS = {}
DATA = {}

for stage in ['1', '2']:
    for scenario in SCENARIOS:
        key = (stage, scenario)
        print(f'Loading S{stage} {scenario}...', end=' ')
        for split in ['oof', 'test']:
            model, features, class_map, df, X, y = load_model_and_data(
                stage, scenario, split)
            MODELS[key] = {'model': model, 'features': features, 'class_map': class_map}
            DATA[(stage, scenario, split)] = {'df': df, 'X': X, 'y': y}
        n_oof  = len(DATA[(stage, scenario, 'oof')]['X'])
        n_test = len(DATA[(stage, scenario, 'test')]['X'])
        print(f'✅  ({len(features)} feat, OOF={n_oof}, Test={n_test})')

print('\n✅ All models and data loaded')

# 4. SHAP Computation

KernelSHAP approach:
- AutoGluon ensemble → `model.predict_proba` wrapper
- Background: 100 random samples from OOF
- Computed on OOF, cached with `.pkl` to avoid recomputation

## 4.1 Function Definitions

In [ ]:
# CELL 5 — SHAP computation & importance functions

def normalize_shap(shap_vals):
    """Convert KernelSHAP output to a standard format:
    - Binary:     (n_samples, n_features) → as is
    - Multiclass: (n_samples, n_features, n_classes) → list of (n_samples, n_features)
    """
    if isinstance(shap_vals, np.ndarray) and shap_vals.ndim == 3:
        # (n_samples, n_features, n_classes) → transpose → list
        n_classes = shap_vals.shape[2]
        return [shap_vals[:, :, i] for i in range(n_classes)]
    return shap_vals


def compute_shap_values(model, X_background, X_explain, stage, scenario,
                        n_background=50, cache_dir=None):
    # Cache check
    if cache_dir is not None:
        cache_file = cache_dir / f'shap_S{stage}_{scenario}.pkl'
        if cache_file.exists():
            print(f'  → Cache found: {cache_file.name}')
            with open(cache_file, 'rb') as f:
                return normalize_shap(pickle.load(f))

    print(f'  → Computing SHAP: S{stage} {scenario} '
          f'(bg={n_background}, explain={len(X_explain)})...')

    # Background sample
    if len(X_background) > n_background:
        bg_idx = np.random.RandomState(42).choice(
            len(X_background), n_background, replace=False)
        bg = X_background.iloc[bg_idx]
    else:
        bg = X_background.copy()

    cols = list(X_background.columns)
    if stage == '1':
        def predict_fn(X_arr):
            X_df = pd.DataFrame(X_arr, columns=cols)
            proba = model.predict_proba(X_df)
            if isinstance(proba, pd.DataFrame):
                return proba.values[:, 1] if proba.shape[1] == 2 else proba.values
            return proba
    else:
        def predict_fn(X_arr):
            X_df = pd.DataFrame(X_arr, columns=cols)
            proba = model.predict_proba(X_df)
            return proba.values if isinstance(proba, pd.DataFrame) else proba

    explainer = shap.KernelExplainer(predict_fn, bg.values)
    shap_values = explainer.shap_values(X_explain.values, silent=True)

    # 3D → list normalization
    shap_values = normalize_shap(shap_values)

    # Save cache
    if cache_dir is not None:
        cache_file = cache_dir / f'shap_S{stage}_{scenario}.pkl'
        with open(cache_file, 'wb') as f:
            pickle.dump(shap_values, f)
        print(f'  → Cache saved: {cache_file.name}')

    return shap_values


def get_shap_importance(shap_vals, feature_names, class_names=None):
    # Safety: 3D → list
    shap_vals = normalize_shap(shap_vals)

    if isinstance(shap_vals, list):
        global_imp = np.mean(
            [np.abs(sv).mean(axis=0) for sv in shap_vals], axis=0)
        df_imp = pd.DataFrame({
            'feature': feature_names, 'mean_abs_shap': global_imp})
        if class_names is not None:
            for i, cn in enumerate(class_names):
                df_imp[f'shap_{cn}'] = np.abs(shap_vals[i]).mean(axis=0)
    else:
        global_imp = np.abs(shap_vals).mean(axis=0)
        df_imp = pd.DataFrame({
            'feature': feature_names, 'mean_abs_shap': global_imp})

    df_imp = df_imp.sort_values(
        'mean_abs_shap', ascending=False).reset_index(drop=True)
    df_imp['rank'] = range(1, len(df_imp) + 1)
    df_imp['feature_display'] = df_imp['feature'].map(feat_display)
    return df_imp

print('✅ SHAP functions defined')

## 4.2 OOF SHAP (4 Model)

In [ ]:
# CELL 6 — Compute SHAP on OOF (sampled, cache-supported)

SHAP_VALUES = {}
SHAP_IMP    = {}
EXPLAIN_IDX = {}
np.random.seed(42)

N_EXPLAIN = 200  # number of OOF samples to explain

for stage in ['1', '2']:
    for scenario in SCENARIOS:
        key = (stage, scenario)
        print(f'\n{"="*60}')
        print(f'S{stage} {scenario}')
        print(f'{"="*60}')

        model     = MODELS[key]['model']
        features  = MODELS[key]['features']
        class_map = MODELS[key]['class_map']
        X_oof     = DATA[(stage, scenario, 'oof')]['X']

        # Samples to explain
        n_explain = min(N_EXPLAIN, len(X_oof))
        explain_idx = np.random.RandomState(42).choice(
            len(X_oof), n_explain, replace=False)
        X_explain = X_oof.iloc[explain_idx]
        EXPLAIN_IDX[key] = explain_idx

        # Compute SHAP (or load from cache)
        shap_vals = compute_shap_values(
            model, X_oof, X_explain, stage, scenario,
            n_background=50, cache_dir=SHAP_DIR)

        # Normalize and store
        shap_vals = normalize_shap(shap_vals)
        SHAP_VALUES[key] = shap_vals

        # Compute importance
        class_names = None
        if class_map is not None:
            class_names = [class_map['class_names'][i]
                           for i in sorted(class_map['class_names'].keys())]

        imp_df = get_shap_importance(shap_vals, features, class_names)
        SHAP_IMP[key] = imp_df

        print(f'\n  Top-5 Features (Mean |SHAP|):')
        for _, row in imp_df.head(5).iterrows():
            print(f'    {row["rank"]}. {row["feature_display"]:25s} → '
                  f'{row["mean_abs_shap"]:.4f}')

print('\n✅ All OOF SHAP values ready')

## 4.3 Test SHAP (Validation)

In [ ]:
# CELL 7 — Use OOF SHAP instead of test SHAP (for misclassification analysis)
# KernelSHAP test set computation is too slow — OOF is sufficient

SHAP_TEST = {}

for stage in ['1', '2']:
    for scenario in SCENARIOS:
        key = (stage, scenario)

        # Use OOF SHAP directly
        SHAP_TEST[key] = SHAP_VALUES[key]
        print(f'S{stage} {scenario}: OOF SHAP will be used')

# In misclassification cells we will use oof instead of DATA test
# Build reference dict
SHAP_TEST_DATA_SPLIT = 'oof'

print('\n✅ SHAP_TEST = OOF SHAP (test computation skipped)')

# 5A — Global Feature Importance

## 5A.1 Beeswarm Summary (2×2)

In [ ]:
# CELL 8 — SHAP Beeswarm Summary 2×2 (improved)

fig, axes = plt.subplots(2, 2, figsize=(18, 16))
fig.suptitle('SHAP Summary (Beeswarm) — All Models',
             fontsize=16, fontweight='bold', y=0.98)

for key, (r, c) in PANEL_POS.items():
    stage, scenario = key
    ax = axes[r, c]
    features  = MODELS[key]['features']
    X_plot    = DATA[(stage, scenario, 'oof')]['X'].iloc[EXPLAIN_IDX[key]]
    shap_vals = SHAP_VALUES[key]

    # S2 multiclass: use the most dominant class's SHAP (direction preserved)
    if isinstance(shap_vals, list):
        # SHAP of the highest-probability class per sample
        sv_stack = np.array(shap_vals)  # (n_classes, n_samples, n_features)
        # Use mean abs for global importance (for ranking)
        mean_imp = np.mean([np.abs(sv).mean(axis=0) for sv in shap_vals], axis=0)
        # For plot: class 0 (DEA) SHAP — preserve direction
        # Alternative: class with highest abs effect
        abs_mean_per_class = np.array([np.abs(sv).mean() for sv in shap_vals])
        dominant_class = np.argmax(abs_mean_per_class)
        sv_plot = shap_vals[dominant_class]
    else:
        mean_imp = np.abs(shap_vals).mean(axis=0)
        sv_plot = shap_vals

    # Top-15
    top_idx = np.argsort(mean_imp)[-15:]

    for j_pos, fi in enumerate(top_idx):
        vals      = sv_plot[:, fi]
        feat_vals = X_plot.values[:, fi]
        fmin, fmax = np.nanpercentile(feat_vals, [2, 98])
        span = fmax - fmin
        normed = (np.clip((feat_vals - fmin) / span, 0, 1)
                  if span > 1e-10 else np.full_like(feat_vals, 0.5))
        jitter = np.random.RandomState(fi).normal(0, 0.15, size=len(vals))
        ax.scatter(vals, j_pos + jitter, c=normed, cmap='coolwarm',
                   s=8, alpha=0.6, rasterized=True)

    display_names = feat_display_list(features)
    ax.set_yticks(range(len(top_idx)))
    ax.set_yticklabels([display_names[i] for i in top_idx], fontsize=9)
    ax.set_xlabel('SHAP value', fontsize=10)
    title = PANEL_TITLES[key]
    if isinstance(shap_vals, list):
        class_map = MODELS[key]['class_map']
        cname = class_map['class_names'][dominant_class]
        title += f'\n(most influential class: {cname})'
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axvline(0, color='grey', lw=0.5, ls='--')
    despine_all(ax)

plt.tight_layout(rect=[0, 0, 1, 0.96])
save_fig(fig, PLOT_DIR, 'shap_summary_beeswarm_2x2')
plt.show()
print('✅ Figure: SHAP Summary Beeswarm')

## 5A.2 Bar Chart — Global Importance (2×2)

In [ ]:
# CELL 9 — Mean |SHAP| bar chart 2×2

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle('SHAP Feature Importance (Mean |SHAP|) — Top 15',
             fontsize=16, fontweight='bold', y=0.98)

for key, (r, c) in PANEL_POS.items():
    stage, scenario = key
    ax  = axes[r, c]
    imp = SHAP_IMP[key].head(15).copy()
    color = SCENARIO_COLORS[scenario]
    y_pos = range(len(imp) - 1, -1, -1)
    ax.barh(list(y_pos), imp['mean_abs_shap'].values,
            color=color, edgecolor='white', height=0.7)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(imp['feature_display'].values, fontsize=9)
    ax.set_xlabel('Mean |SHAP|', fontsize=10)
    ax.set_title(PANEL_TITLES[key], fontsize=12, fontweight='bold')
    despine_all(ax)

plt.tight_layout(rect=[0, 0, 1, 0.96])
save_fig(fig, PLOT_DIR, 'shap_bar_importance_2x2')
plt.show()
print('✅ Figure: SHAP Bar Importance')

## 5A.3 Scenario Comparison

In [ ]:
# CELL 10 — CBC_Only vs CBC_BIO

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle('SHAP Importance: CBC Only vs CBC + Biochemistry',
             fontsize=14, fontweight='bold')

for idx, stage in enumerate(['1', '2']):
    ax = axes[idx]
    imp_cbc = SHAP_IMP[(stage, 'CBC_Only')].set_index('feature')
    imp_bio = SHAP_IMP[(stage, 'CBC_BIO')].set_index('feature')
    common = list(set(imp_cbc.index) & set(imp_bio.index))
    if not common:
        ax.text(0.5, 0.5, 'No common feature', ha='center', va='center',
                transform=ax.transAxes)
        continue
    top_bio = imp_bio.loc[imp_bio.index.isin(common)].head(10).index.tolist()
    y_pos = range(len(top_bio) - 1, -1, -1)
    vals_cbc = [imp_cbc.loc[f, 'mean_abs_shap'] if f in imp_cbc.index else 0
                for f in top_bio]
    vals_bio = [imp_bio.loc[f, 'mean_abs_shap'] for f in top_bio]
    labels = [feat_display(f) for f in top_bio]
    ax.barh([y + 0.2 for y in y_pos], vals_cbc, height=0.35,
            color=SCENARIO_COLORS['CBC_Only'], label='CBC Only', edgecolor='white')
    ax.barh([y - 0.2 for y in y_pos], vals_bio, height=0.35,
            color=SCENARIO_COLORS['CBC_BIO'], label='CBC+BIO', edgecolor='white')
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Mean |SHAP|', fontsize=10)
    ax.set_title(f'Stage {stage}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, loc='lower right')
    despine_all(ax)

plt.tight_layout()
save_fig(fig, PLOT_DIR, 'shap_scenario_comparison')
plt.show()
print('✅ Figure: Scenario Comparison')

## 5A.4 Tez vs SHAP Ranking

In [ ]:
# CELL 11 — Thesis vs SHAP table + bump chart

comparison_rows = []
for key in [('1','CBC_Only'),('1','CBC_BIO'),('2','CBC_Only'),('2','CBC_BIO')]:
    stage, scenario = key
    thesis_top5 = THESIS_IMPORTANCE[key]
    shap_imp    = SHAP_IMP[key]
    shap_top5   = shap_imp.head(5)['feature'].tolist()
    for rank, (t_feat, s_feat) in enumerate(zip(thesis_top5, shap_top5), 1):
        sr = shap_imp[shap_imp['feature'] == t_feat]['rank'].values
        comparison_rows.append({
            'Stage': f'S{stage}', 'Scenario': scenario,
            'Thesis_Rank': rank, 'Thesis_Feature': feat_display(t_feat),
            'SHAP_Rank': rank, 'SHAP_Feature': feat_display(s_feat),
            'Thesis_in_SHAP_Rank': sr[0] if len(sr) > 0 else '-',
        })
df_comparison = pd.DataFrame(comparison_rows)
print(df_comparison.to_string(index=False))

# --- Bump chart ---
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Feature Importance Ranking: Thesis (AutoGluon) vs SHAP',
             fontsize=14, fontweight='bold', y=0.98)

for key, (r, c) in PANEL_POS.items():
    stage, scenario = key
    ax = axes[r, c]
    thesis_top5 = THESIS_IMPORTANCE[key]
    shap_imp = SHAP_IMP[key]
    names, t_ranks, s_ranks = [], [], []
    for i, feat in enumerate(thesis_top5):
        names.append(feat_display(feat))
        t_ranks.append(i + 1)
        sr = shap_imp[shap_imp['feature'] == feat]['rank'].values
        s_ranks.append(sr[0] if len(sr) > 0 else len(shap_imp))
    for i in range(len(names)):
        color = PALETTE['highlight'] if s_ranks[i] > 5 else PALETTE['accent1']
        ax.plot([0, 1], [t_ranks[i], s_ranks[i]],
                'o-', color=color, lw=2, markersize=8, zorder=3)
        ax.text(-0.05, t_ranks[i], names[i], ha='right', va='center', fontsize=8)
        ax.text(1.05, s_ranks[i], f'#{s_ranks[i]}',
                ha='left', va='center', fontsize=8, color=color)
    ax.set_xlim(-0.4, 1.4)
    ax.set_ylim(max(max(s_ranks), 5) + 1, 0)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Thesis\n(AutoGluon)', 'SHAP\n(KernelSHAP)'], fontsize=10)
    ax.set_ylabel('Rank', fontsize=10)
    ax.set_title(PANEL_TITLES[key], fontsize=11, fontweight='bold')
    ax.axhline(5.5, color='grey', lw=0.5, ls=':', alpha=0.5)
    despine_all(ax)

plt.tight_layout(rect=[0, 0, 1, 0.96])
save_fig(fig, PLOT_DIR, 'shap_vs_thesis_ranking')
plt.show()
print('✅ Figure: Thesis vs SHAP Ranking')

# 5B — Per-Class SHAP (Stage 2)

## 5B.1 Per-Class Importance

In [ ]:
# CELL 12 — Per-class SHAP bar (S2, 4 classes × 2 scenarios)

# Internal class key → display label (for titles only; CLASS_COLORS_S2 keys stay internal)
CLASS_DISP_S2 = {'DEA': 'IDA', 'HA': 'HA', 'HGB HTZ': 'HGB HTZ', 'Normal': 'Normal'}

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
fig.suptitle('Per-Class SHAP Feature Importance (Stage 2, Top-10)',
             fontsize=14, fontweight='bold', y=0.98)

for s_idx, scenario in enumerate(SCENARIOS):
    key = ('2', scenario)
    shap_vals  = SHAP_VALUES[key]
    features   = MODELS[key]['features']
    class_map  = MODELS[key]['class_map']
    class_names = [class_map['class_names'][i]
                   for i in sorted(class_map['class_names'].keys())]
    for c_idx, cname in enumerate(class_names):
        ax = axes[s_idx, c_idx]
        sv = shap_vals[c_idx]
        mean_abs = np.abs(sv).mean(axis=0)
        top_idx  = np.argsort(mean_abs)[-10:][::-1]
        top_names = [feat_display(features[i]) for i in top_idx]
        top_vals  = mean_abs[top_idx]
        color = CLASS_COLORS_S2.get(cname, PALETTE['base1'])
        y_pos = range(len(top_names) - 1, -1, -1)
        ax.barh(list(y_pos), top_vals, color=color, edgecolor='white', height=0.7)
        ax.set_yticks(list(y_pos))
        ax.set_yticklabels(top_names, fontsize=8)
        ax.set_xlabel('Mean |SHAP|', fontsize=9)
        prefix = 'CBC' if scenario == 'CBC_Only' else 'BIO'
        ax.set_title(f'{prefix} — {CLASS_DISP_S2.get(cname, cname)}', fontsize=10, fontweight='bold')
        despine_all(ax)

plt.tight_layout(rect=[0, 0, 1, 0.95])
save_fig(fig, PLOT_DIR, 'shap_per_class_s2')
plt.show()
print('✅ Figure: Per-Class SHAP (S2)')

## 5B.2 DEA vs HGB_HTZ SHAP Profili

In [ ]:
# CELL 13 — DEA vs HGB_HTZ

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('SHAP Profile: IDA vs HGB HTZ',
             fontsize=14, fontweight='bold')

for idx, scenario in enumerate(SCENARIOS):
    ax  = axes[idx]
    key = ('2', scenario)
    shap_vals = SHAP_VALUES[key]
    features  = MODELS[key]['features']
    class_map = MODELS[key]['class_map']
    dea_idx = [k for k, v in class_map['class_names'].items() if v == 'DEA'][0]
    hgb_idx = [k for k, v in class_map['class_names'].items() if v == 'HGB HTZ'][0]
    dea_imp = np.abs(shap_vals[dea_idx]).mean(axis=0)
    hgb_imp = np.abs(shap_vals[hgb_idx]).mean(axis=0)
    diff    = dea_imp - hgb_imp
    top_idx  = np.argsort(np.abs(diff))[-10:]
    sort_idx = top_idx[np.argsort(diff[top_idx])]
    names = [feat_display(features[i]) for i in sort_idx]
    vals  = diff[sort_idx]
    colors = [PALETTE['highlight'] if d > 0 else PALETTE['accent1'] for d in vals]
    ax.barh(range(len(names)), vals, color=colors, edgecolor='white', height=0.7)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel('Δ SHAP (IDA − HGB HTZ)', fontsize=10)
    ax.axvline(0, color='grey', lw=0.5, ls='--')
    ax.set_title(scenario.replace('_', ' '), fontsize=12, fontweight='bold')
    legend_el = [Patch(facecolor=PALETTE['highlight'], label='IDA > HGB HTZ'),
                 Patch(facecolor=PALETTE['accent1'],   label='HGB HTZ > IDA')]
    ax.legend(handles=legend_el, fontsize=8, loc='lower right')
    despine_all(ax)

plt.tight_layout()
save_fig(fig, PLOT_DIR, 'shap_dea_vs_hgbhtz')
plt.show()
print('✅ Figure: IDA vs HGB HTZ SHAP')

## 5B.3 Biochemistry Feature SHAP Contribution

In [ ]:
# CELL 14 — Biochemistry contribution

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle('Biochemistry Feature SHAP Contribution (CBC + BIO)',
             fontsize=14, fontweight='bold')

for idx, stage in enumerate(['1', '2']):
    ax  = axes[idx]
    key = (stage, 'CBC_BIO')
    shap_vals = SHAP_VALUES[key]
    features  = MODELS[key]['features']
    bio_in_model = [f for f in features if f in BIO_FEATURES]

    if isinstance(shap_vals, list):
        sv_all = np.mean([np.abs(sv) for sv in shap_vals], axis=0)
    else:
        sv_all = np.abs(shap_vals)

    bio_idx = [features.index(f) for f in bio_in_model]
    cbc_idx = [i for i in range(len(features)) if i not in bio_idx]
    bio_mean = sv_all[:, bio_idx].mean(axis=1)
    cbc_mean = sv_all[:, cbc_idx].mean(axis=1)

    parts = ax.violinplot([cbc_mean, bio_mean], positions=[0, 1],
                          showmeans=True, showmedians=True)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor([PALETTE['base1'], PALETTE['accent3']][i])
        pc.set_alpha(0.6)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['CBC\nFeatures', 'Biochemistry\nFeatures'], fontsize=11)
    ax.set_ylabel('Mean |SHAP| / sample', fontsize=10)
    ax.set_title(f'Stage {stage}', fontsize=12, fontweight='bold')
    total = sv_all.mean()
    bio_pct = sv_all[:, bio_idx].mean() / total * 100 if total > 0 else 0
    ax.text(0.95, 0.95, f'BIO contribution: {bio_pct:.1f}%',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    despine_all(ax)

plt.tight_layout()
save_fig(fig, PLOT_DIR, 'shap_bio_contribution')
plt.show()
print('✅ Figure: Biochemistry SHAP Contribution')

# 5C — SHAP Dependence Plots

Top-5 features per model, color = interaction feature.

In [ ]:
# CELL 15 — Dependence plots (improved)

from scipy.ndimage import uniform_filter1d

for scenario in SCENARIOS:
    for stage in ['1', '2']:
        key = (stage, scenario)
        features  = MODELS[key]['features']
        shap_vals = SHAP_VALUES[key]
        X_plot    = DATA[(stage, scenario, 'oof')]['X'].iloc[EXPLAIN_IDX[key]]

        if isinstance(shap_vals, list):
            mean_imp   = np.mean([np.abs(sv).mean(axis=0) for sv in shap_vals], axis=0)
            sv_for_dep = np.mean(np.array(shap_vals), axis=0)
        else:
            mean_imp   = np.abs(shap_vals).mean(axis=0)
            sv_for_dep = shap_vals

        top5_idx = np.argsort(mean_imp)[-5:][::-1]
        fig, axes_row = plt.subplots(1, 5, figsize=(25, 5))
        fig.suptitle(f'SHAP Dependence — S{stage} {scenario.replace("_"," ")} (Top 5)',
                     fontsize=13, fontweight='bold')

        for j, f_idx in enumerate(top5_idx):
            ax     = axes_row[j]
            f_disp = feat_display(features[f_idx])
            x_vals = X_plot.iloc[:, f_idx].values
            s_vals = sv_for_dep[:, f_idx]

            # Outlier clipping (1-99 percentile)
            xp1, xp99 = np.nanpercentile(x_vals, [1, 99])
            mask = (x_vals >= xp1) & (x_vals <= xp99) & ~np.isnan(x_vals)
            x_clean = x_vals[mask]
            s_clean = s_vals[mask]

            other_idx = [i for i in top5_idx if i != f_idx]
            if other_idx:
                corrs = [abs(np.corrcoef(
                    s_clean, X_plot.iloc[:, oi].fillna(0).values[mask])[0, 1])
                    for oi in other_idx]
                best_other = other_idx[np.argmax(corrs)]
                c_vals = X_plot.iloc[:, best_other].values[mask]
                cmin, cmax = np.nanpercentile(c_vals, [2, 98])
                span = cmax - cmin
                normed_c = (np.clip((c_vals - cmin) / span, 0, 1)
                            if span > 1e-10 else np.full_like(c_vals, 0.5))
                ax.scatter(x_clean, s_clean, c=normed_c, cmap='coolwarm',
                           s=25, alpha=0.7, edgecolors='white', linewidths=0.3,
                           rasterized=True)
                if j == 4:
                    cbar = plt.colorbar(
                        ax.collections[0], ax=ax, fraction=0.05)
                    cbar.set_label(feat_display(features[best_other]), fontsize=8)
            else:
                ax.scatter(x_clean, s_clean, c=PALETTE['base1'],
                           s=25, alpha=0.7, edgecolors='white', linewidths=0.3,
                           rasterized=True)

            # Trend line (LOWESS-like — running mean)
            if len(x_clean) > 20:
                sort_idx = np.argsort(x_clean)
                x_sorted = x_clean[sort_idx]
                s_sorted = s_clean[sort_idx]
                window = max(len(x_clean) // 10, 5)
                s_smooth = uniform_filter1d(s_sorted.astype(float), size=window)
                ax.plot(x_sorted, s_smooth, color=PALETTE['highlight'],
                        lw=2, alpha=0.8, zorder=5)

            ax.axhline(0, color='grey', lw=0.5, ls='--')
            ax.set_xlabel(f_disp, fontsize=9)
            ax.set_ylabel('SHAP value' if j == 0 else '', fontsize=9)
            ax.set_title(f_disp, fontsize=10, fontweight='bold')
            despine_all(ax)

        plt.tight_layout()
        save_fig(fig, PLOT_DIR, f'shap_dependence_S{stage}_{scenario}')
        plt.show()

print('✅ Figure: All Dependence Plots')

# 5D — Misclassification SHAP

## 5D.1 Correct vs Wrong Profile

In [ ]:
# CELL 16 — Correct vs wrong prediction SHAP (on OOF)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('SHAP Profile: Correct vs Wrong Prediction (OOF)',
             fontsize=14, fontweight='bold', y=0.98)
misclass_rows = []

for key, (r, c) in PANEL_POS.items():
    stage, scenario = key
    ax = axes[r, c]
    df_oof    = DATA[(stage, scenario, 'oof')]['df']
    features  = MODELS[key]['features']
    shap_vals = SHAP_VALUES[key]
    idx       = EXPLAIN_IDX[key]

    # Same sample as SHAP
    df_sub       = df_oof.iloc[idx]
    correct_mask = df_sub['correct'].values.astype(bool)
    wrong_mask   = ~correct_mask

    if isinstance(shap_vals, list):
        sv_abs = np.mean([np.abs(sv) for sv in shap_vals], axis=0)
    else:
        sv_abs = np.abs(shap_vals)

    mean_correct = sv_abs[correct_mask].mean(axis=0) if correct_mask.sum() > 0 else np.zeros(len(features))
    mean_wrong   = sv_abs[wrong_mask].mean(axis=0) if wrong_mask.sum() > 0 else np.zeros(len(features))
    diff     = mean_wrong - mean_correct
    top_idx  = np.argsort(np.abs(diff))[-10:]
    sort_idx = top_idx[np.argsort(diff[top_idx])]
    names = [feat_display(features[i]) for i in sort_idx]
    vals  = diff[sort_idx]

    colors = [PALETTE['highlight'] if d > 0 else PALETTE['accent1'] for d in vals]
    ax.barh(range(len(names)), vals, color=colors, edgecolor='white', height=0.7)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.axvline(0, color='grey', lw=0.5, ls='--')
    ax.set_xlabel('Δ Mean |SHAP| (Wrong − Correct)', fontsize=9)
    ax.set_title(f'{PANEL_TITLES[key]}\n(Correct={correct_mask.sum()}, Wrong={wrong_mask.sum()})',
                 fontsize=10, fontweight='bold')
    despine_all(ax)
    for i in sort_idx:
        misclass_rows.append({'Stage': f'S{stage}', 'Scenario': scenario,
            'Feature': feat_display(features[i]),
            'Mean_SHAP_Correct': mean_correct[i],
            'Mean_SHAP_Wrong': mean_wrong[i], 'Delta': diff[i]})

plt.tight_layout(rect=[0, 0, 1, 0.96])
save_fig(fig, PLOT_DIR, 'shap_misclassification_profile')
plt.show()
df_misclass = pd.DataFrame(misclass_rows)
print('✅ Figure: Misclassification SHAP')

## 5D.2 HGB_HTZ → DEA Error

In [ ]:
# CELL 17 — HGB_HTZ → DEA error (on OOF)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('SHAP: HGB HTZ → IDA Error (S2 OOF)', fontsize=14, fontweight='bold')

for idx_s, scenario in enumerate(SCENARIOS):
    ax  = axes[idx_s]
    key = ('2', scenario)
    df_oof     = DATA[('2', scenario, 'oof')]['df']
    features   = MODELS[key]['features']
    class_map  = MODELS[key]['class_map']
    shap_vals  = SHAP_VALUES[key]
    idx        = EXPLAIN_IDX[key]

    df_sub = df_oof.iloc[idx]

    if 'target_label' in df_sub.columns:
        hgb_mask = df_sub['target_label'] == 'HGB HTZ'
    else:
        hgb_target = [k for k, v in class_map['class_names'].items() if v == 'HGB HTZ'][0]
        hgb_mask = df_sub['target'] == hgb_target

    dea_target = [k for k, v in class_map['class_names'].items() if v == 'DEA'][0]
    error_mask  = hgb_mask & (df_sub['pred'] == dea_target)
    correct_hgb = hgb_mask & df_sub['correct'].astype(bool)
    n_error, n_correct = error_mask.sum(), correct_hgb.sum()

    if n_error == 0:
        ax.text(0.5, 0.5, f'No HGB HTZ→IDA error\n({scenario})',
                ha='center', va='center', transform=ax.transAxes)
        despine_all(ax)
        continue

    sv_dea = shap_vals[dea_target]
    error_shap   = np.abs(sv_dea[error_mask.values]).mean(axis=0)
    correct_shap = np.abs(sv_dea[correct_hgb.values]).mean(axis=0) if n_correct > 0 else np.zeros(len(features))
    diff     = error_shap - correct_shap
    top_idx  = np.argsort(np.abs(diff))[-10:]
    sort_idx = top_idx[np.argsort(diff[top_idx])]
    names = [feat_display(features[i]) for i in sort_idx]
    vals  = diff[sort_idx]

    colors = [PALETTE['highlight'] if d > 0 else PALETTE['accent1'] for d in vals]
    ax.barh(range(len(names)), vals, color=colors, edgecolor='white', height=0.7)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.axvline(0, color='grey', lw=0.5, ls='--')
    ax.set_xlabel('Δ |SHAP for IDA| (Error − Correct HGB HTZ)', fontsize=9)
    ax.set_title(f'{scenario.replace("_"," ")}\nError={n_error}, Correct={n_correct}',
                 fontsize=11, fontweight='bold')
    despine_all(ax)

plt.tight_layout()
save_fig(fig, PLOT_DIR, 'shap_hgbhtz_to_dea_error')
plt.show()
print('✅ Figure: HGB HTZ→IDA Error SHAP')

## 5D.3 Lost Patients (S1: IAS → DAS)

In [ ]:
# CELL 18 — Lost patients (on OOF)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('SHAP Profile of Lost Patients (S1: AAC → OAC Error)',
             fontsize=14, fontweight='bold')

for idx_s, scenario in enumerate(SCENARIOS):
    ax  = axes[idx_s]
    key = ('1', scenario)
    df_oof    = DATA[('1', scenario, 'oof')]['df']
    features  = MODELS[key]['features']
    shap_vals = SHAP_VALUES[key]
    idx       = EXPLAIN_IDX[key]

    df_sub = df_oof.iloc[idx]

    lost_mask   = (df_sub['target'] == 1) & (df_sub['pred'] == 0)
    correct_ias = (df_sub['target'] == 1) & df_sub['correct'].astype(bool)
    n_lost, n_correct = lost_mask.sum(), correct_ias.sum()

    if n_lost == 0:
        ax.text(0.5, 0.5, f'No lost patients ({scenario})',
                ha='center', va='center', transform=ax.transAxes)
        despine_all(ax)
        continue

    sv = shap_vals
    lost_shap    = np.abs(sv[lost_mask.values]).mean(axis=0)
    correct_shap = np.abs(sv[correct_ias.values]).mean(axis=0) if n_correct > 0 else np.zeros(len(features))
    diff     = lost_shap - correct_shap
    top_idx  = np.argsort(np.abs(diff))[-10:]
    sort_idx = top_idx[np.argsort(diff[top_idx])]
    names = [feat_display(features[i]) for i in sort_idx]
    vals  = diff[sort_idx]

    colors = [PALETTE['highlight'] if d > 0 else PALETTE['accent1'] for d in vals]
    ax.barh(range(len(names)), vals, color=colors, edgecolor='white', height=0.7)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.axvline(0, color='grey', lw=0.5, ls='--')
    ax.set_xlabel('Δ |SHAP| (Lost − Correct AAC)', fontsize=9)
    ax.set_title(f'{scenario.replace("_"," ")}\nLost={n_lost}, Correct AAC={n_correct}',
                 fontsize=11, fontweight='bold')
    despine_all(ax)

plt.tight_layout()
save_fig(fig, PLOT_DIR, 'shap_lost_patients_s1')
plt.show()
print('✅ Figure: Lost Patients SHAP')

# 5E — Clinical Interpretation & Outputs

## 5E.1 Excel: Global Importance

In [ ]:
# CELL 19 — shap_global_importance.xlsx

rows = []
for key in [('1','CBC_Only'),('1','CBC_BIO'),('2','CBC_Only'),('2','CBC_BIO')]:
    stage, scenario = key
    imp = SHAP_IMP[key].copy()
    imp.insert(0, 'Stage', f'S{stage}')
    imp.insert(1, 'Scenario', scenario)
    rows.append(imp)
df_global = pd.concat(rows, ignore_index=True)
out = ANALYSIS_DIR / 'shap_global_importance.xlsx'
df_global.to_excel(out, index=False)
print(f'✅ {out.name}  ({len(df_global)} rows)')

## 5E.2 Excel: Per-Class SHAP

In [ ]:
# CELL 20 — shap_per_class.xlsx

rows = []
for scenario in SCENARIOS:
    imp = SHAP_IMP[('2', scenario)].copy()
    imp.insert(0, 'Scenario', scenario)
    rows.append(imp)
df_pc = pd.concat(rows, ignore_index=True)
out = ANALYSIS_DIR / 'shap_per_class.xlsx'
df_pc.to_excel(out, index=False)
print(f'✅ {out.name}')

## 5E.3 Excel: Misclassification

In [ ]:
# CELL 21 — shap_misclassification.xlsx

out = ANALYSIS_DIR / 'shap_misclassification.xlsx'
df_misclass.to_excel(out, index=False)
print(f'✅ {out.name}  ({len(df_misclass)} rows)')

## 5E.4 Excel: Biochemistry Contribution

In [ ]:
# CELL 22 — shap_bio_contribution.xlsx

bio_rows = []
for stage in ['1', '2']:
    key = (stage, 'CBC_BIO')
    features = MODELS[key]['features']
    shap_vals = SHAP_VALUES[key]
    sv_abs = np.mean([np.abs(sv) for sv in shap_vals], axis=0) if isinstance(shap_vals, list) else np.abs(shap_vals)
    for i, feat in enumerate(features):
        is_bio = feat in BIO_FEATURES
        bio_rows.append({'Stage': f'S{stage}', 'Feature': feat,
            'Feature_Display': feat_display(feat),
            'Mean_Abs_SHAP': sv_abs[:, i].mean(),
            'Is_Biochemistry': is_bio,
            'Category': 'Biochemistry' if is_bio else 'CBC'})

df_bio = pd.DataFrame(bio_rows).sort_values(['Stage', 'Mean_Abs_SHAP'], ascending=[True, False])

summary_rows = []
for stage in ['1', '2']:
    sub = df_bio[df_bio['Stage'] == f'S{stage}']
    bio_s, cbc_s = sub[sub['Is_Biochemistry']], sub[~sub['Is_Biochemistry']]
    total = sub['Mean_Abs_SHAP'].sum()
    summary_rows.append({'Stage': f'S{stage}',
        'CBC_Total_SHAP': cbc_s['Mean_Abs_SHAP'].sum(),
        'BIO_Total_SHAP': bio_s['Mean_Abs_SHAP'].sum(),
        'BIO_Percentage': bio_s['Mean_Abs_SHAP'].sum() / total * 100 if total > 0 else 0,
        'N_CBC_Features': len(cbc_s), 'N_BIO_Features': len(bio_s)})
df_bio_summary = pd.DataFrame(summary_rows)

out = ANALYSIS_DIR / 'shap_bio_contribution.xlsx'
with pd.ExcelWriter(out) as w:
    df_bio.to_excel(w, sheet_name='Per_Feature', index=False)
    df_bio_summary.to_excel(w, sheet_name='Summary', index=False)
print(f'✅ {out.name}')
print('\nBiochemistry SHAP Contribution Summary:')
print(df_bio_summary.to_string(index=False))

## 5E.5 Excel: Tez vs SHAP

In [ ]:
# CELL 23 — shap_vs_thesis_ranking.xlsx

out = ANALYSIS_DIR / 'shap_vs_thesis_ranking.xlsx'
df_comparison.to_excel(out, index=False)
print(f'✅ {out.name}')

## 5E.6 Klinik Yorumlama Raporu

In [ ]:
# CELL 24 — Clinical interpretation report

lines = []
lines.append('=' * 80)
lines.append('M5 — SHAP EXPLAINABILITY REPORT')
lines.append('Anemia CDS Project — Clinical Interpretation')
lines.append('=' * 80)

for key in [('1','CBC_Only'),('1','CBC_BIO'),('2','CBC_Only'),('2','CBC_BIO')]:
    stage, scenario = key
    lines.append(f'\n{chr(9472)*60}')
    lines.append(f'Stage {stage} — {scenario.replace(chr(95), chr(32))}')
    lines.append(f'{chr(9472)*60}')
    imp = SHAP_IMP[key]
    lines.append('\nTop-10 Feature Importance (SHAP):')
    for _, row in imp.head(10).iterrows():
        lines.append(f'  {row["rank"]:2d}. {row["feature_display"]:25s}  '
                     f'Mean|SHAP|={row["mean_abs_shap"]:.4f}')
    thesis_feats = THESIS_IMPORTANCE[key]
    lines.append(f'\nThesis Top-5 vs SHAP:')
    for i, tf in enumerate(thesis_feats, 1):
        sr = imp[imp['feature'] == tf]['rank'].values
        sr_str = str(sr[0]) if len(sr) > 0 else 'N/A'
        lines.append(f'  Thesis #{i}: {feat_display(tf):25s} -> SHAP #{sr_str}')

lines.append(f'\n{chr(9472)*60}')
lines.append('BIOCHEMISTRY CONTRIBUTION')
lines.append(f'{chr(9472)*60}')
for _, row in df_bio_summary.iterrows():
    lines.append(f'\n  {row["Stage"]}:')
    lines.append(f'    CBC total SHAP : {row["CBC_Total_SHAP"]:.4f} ({row["N_CBC_Features"]} features)')
    lines.append(f'    BIO total SHAP : {row["BIO_Total_SHAP"]:.4f} ({row["N_BIO_Features"]} features)')
    lines.append(f'    BIO contribution: {row["BIO_Percentage"]:.1f}%')

report_text = '\n'.join(lines)
report_path = REPORT_DIR / 'shap_clinical_report.txt'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_text)
print(report_text)
print(f'\n✅ Report saved: {report_path}')

# 10. Create src/m5_shap.py

In [ ]:
# CELL 25 — create m5_shap.py module

m5_path = PATHS.src / 'm5_shap.py'

m5_lines = [
    '"""',
    'm5_shap.py — SHAP Explainability Functions',
    'Anemia CDS Project — Module 5',
    '"""',
    '',
    'import numpy as np',
    'import pandas as pd',
    'import shap',
    'import pickle',
    'import joblib',
    'from pathlib import Path',
    'from autogluon.tabular import TabularPredictor',
    '',
    f'FEATURE_DISPLAY = {FEATURE_DISPLAY!r}',
    '',
    f'BIO_FEATURES = {BIO_FEATURES!r}',
    '',
    '',
    'def feat_display(name):',
    '    return FEATURE_DISPLAY.get(name, name)',
    '',
    '',
    'def normalize_shap(shap_vals):',
    '    """Convert KernelSHAP output to a standard format.',
    '    (n_samples, n_features, n_classes) -> list of (n_samples, n_features)"""',
    '    if isinstance(shap_vals, np.ndarray) and shap_vals.ndim == 3:',
    '        return [shap_vals[:, :, i] for i in range(shap_vals.shape[2])]',
    '    return shap_vals',
    '',
    '',
    'def load_model_for_shap(frozen_models_dir, stage, scenario):',
    '    pattern = f"*Stage{stage}_{scenario}*"',
    '    model_dir = list(Path(frozen_models_dir).glob(pattern))[0]',
    '    model = TabularPredictor.load(str(model_dir / "AutoGluon_Model"))',
    '    features = joblib.load(model_dir / "feature_names.joblib")',
    '    class_map = None',
    '    cm_path = model_dir / "class_mapping.joblib"',
    '    if cm_path.exists():',
    '        class_map = joblib.load(cm_path)',
    '    return model, features, class_map',
    '',
    '',
    'def compute_shap_values(model, X_background, X_explain, stage,',
    '                        n_background=50, cache_path=None):',
    '    if cache_path is not None and Path(cache_path).exists():',
    '        with open(cache_path, "rb") as f:',
    '            return normalize_shap(pickle.load(f))',
    '    if len(X_background) > n_background:',
    '        bg_idx = np.random.RandomState(42).choice(',
    '            len(X_background), n_background, replace=False)',
    '        bg = X_background.iloc[bg_idx]',
    '    else:',
    '        bg = X_background.copy()',
    '    cols = list(X_background.columns)',
    '    if stage == "1":',
    '        def predict_fn(X_arr):',
    '            X_df = pd.DataFrame(X_arr, columns=cols)',
    '            proba = model.predict_proba(X_df)',
    '            if isinstance(proba, pd.DataFrame):',
    '                return proba.values[:, 1] if proba.shape[1] == 2 else proba.values',
    '            return proba',
    '    else:',
    '        def predict_fn(X_arr):',
    '            X_df = pd.DataFrame(X_arr, columns=cols)',
    '            proba = model.predict_proba(X_df)',
    '            return proba.values if isinstance(proba, pd.DataFrame) else proba',
    '    explainer = shap.KernelExplainer(predict_fn, bg.values)',
    '    shap_values = explainer.shap_values(X_explain.values, silent=True)',
    '    shap_values = normalize_shap(shap_values)',
    '    if cache_path is not None:',
    '        Path(cache_path).parent.mkdir(parents=True, exist_ok=True)',
    '        with open(cache_path, "wb") as f:',
    '            pickle.dump(shap_values, f)',
    '    return shap_values',
    '',
    '',
    'def get_shap_importance(shap_vals, feature_names, class_names=None):',
    '    shap_vals = normalize_shap(shap_vals)',
    '    if isinstance(shap_vals, list):',
    '        global_imp = np.mean([np.abs(sv).mean(axis=0) for sv in shap_vals], axis=0)',
    '        df_imp = pd.DataFrame({"feature": feature_names, "mean_abs_shap": global_imp})',
    '        if class_names is not None:',
    '            for i, cn in enumerate(class_names):',
    '                df_imp[f"shap_{cn}"] = np.abs(shap_vals[i]).mean(axis=0)',
    '    else:',
    '        global_imp = np.abs(shap_vals).mean(axis=0)',
    '        df_imp = pd.DataFrame({"feature": feature_names, "mean_abs_shap": global_imp})',
    '    df_imp = df_imp.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)',
    '    df_imp["rank"] = range(1, len(df_imp) + 1)',
    '    df_imp["feature_display"] = df_imp["feature"].map(feat_display)',
    '    return df_imp',
    '',
    '',
    'def load_cached_shap(cache_dir, stage, scenario):',
    '    cache_file = Path(cache_dir) / f"shap_S{stage}_{scenario}.pkl"',
    '    if cache_file.exists():',
    '        with open(cache_file, "rb") as f:',
    '            return normalize_shap(pickle.load(f))',
    '    return None',
]

with open(m5_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(m5_lines))

print(f'✅ {m5_path} saved')

# 11. Final Summary

In [ ]:
# CELL 27 — Final Summary

print('=' * 70)
print('M5 SHAP EXPLAINABILITY — COMPLETED')
print('=' * 70)

pngs = list(PLOT_DIR.rglob('*.png'))
tifs = list(PLOT_DIR.rglob('*.tif'))
print(f'\n📊 FIGURES ({len(pngs)} PNG + {len(tifs)} TIFF):')
for p in sorted(pngs):
    print(f'  ✅ {p.name}  ({p.stat().st_size/1024:.0f} KB)')

print(f'\n📋 TABLES:')
for p in sorted(ANALYSIS_DIR.glob('*.xlsx')):
    print(f'  ✅ {p.name}')

print(f'\n💾 SHAP CACHE:')
for p in sorted(SHAP_DIR.glob('*.pkl')):
    print(f'  ✅ {p.name}')

print(f'\n📝 REPORTS:')
for p in sorted(REPORT_DIR.glob('*')):
    print(f'  ✅ {p.name}')

print(f'\n🐍 MODULE: {PATHS.src / "m5_shap.py"}')
print('\n✅ M5 completed — ready to proceed to M7 Final Figures & Tables')

In [ ]:
# m5_shap.py update — AutoGluon lazy import

import re

m5_path = PATHS.src / 'm5_shap.py'
text = m5_path.read_text(encoding='utf-8')

# 1) Remove top-level AutoGluon import
text = text.replace('from autogluon.tabular import TabularPredictor\n', '')

# 2) Add lazy import inside load_model_for_shap
text = text.replace(
    'def load_model_for_shap(frozen_models_dir, stage, scenario):\n'
    '    pattern',
    'def load_model_for_shap(frozen_models_dir, stage, scenario):\n'
    '    from autogluon.tabular import TabularPredictor  # lazy import\n'
    '    pattern'
)

m5_path.write_text(text, encoding='utf-8')
print('✅ m5_shap.py updated — AutoGluon lazy import')
print('   Top-level import removed, loaded only inside load_model_for_shap')